# Git 基本原理与分支

预计阅读时间：10 分钟。

Git 记录的是项目快照（snapshot）。每次提交（commit）都是项目在某个时刻的状态。

## 原理

核心概念：

| 概念 | 说明 |
| --- | --- |
| 仓库 repository | Git 管理的项目目录 |
| 提交 commit | 一次保存下来的项目快照 |
| HEAD | 当前所在位置，通常指向当前分支 |
| 哈希值 | 提交的唯一标识 |

一次提交应该对应一个清楚、可说明的项目状态。

## 快照是什么

可以把一次提交理解成“项目目录的一张快照”。这张快照不是简单复制整个文件夹，而是由 Git 对象组织起来的：

```mermaid
flowchart TD
    C1[commit A] --> T1[tree: 项目根目录]
    T1 --> F1[blob: train.py]
    T1 --> D1[tree: configs/]
    D1 --> F2[blob: configs/base.yaml]

    C2[commit B] --> T2[tree: 项目根目录]
    T2 --> F3[blob: train.py 新版本]
    T2 --> D1
```

其中：

- `commit` 记录一次提交的信息，并指向项目根目录对应的 `tree`；
- `tree` 记录目录结构，以及目录下有哪些文件或子目录；
- `blob` 记录文件内容。

如果某个文件或子目录没有变化，新的提交可以继续引用旧的对象。上图中，`commit B` 修改了 `train.py`，但 `configs/` 没变，所以仍然复用原来的 `configs/` 子树。也因此，Git 的快照模型通常比“每次完整复制一份项目目录”显著节省空间。

## Commit

`commit` 是 Git 中的一次提交对象。它不只是一个文件快照，还包含提交说明、作者、时间、父提交等信息。

其中作者信息可以通过 `git config` 配置：



In [ ]:
git config --global user.name "Your Name"
git config --global user.email "you@example.com"




这两个配置会写入提交记录中的作者信息，适用于所有 Git 仓库，不是 GitHub 专用配置。

每个 `commit` 都会有一个哈希值，例如：

```text
a1b2c3d
```

这个哈希值可以理解为提交的唯一编号。直接记哈希值不方便，所以 Git 提供了一些更容易使用的引用：

- `main`：分支名，通常表示项目的主分支；
- `HEAD`：当前位置，通常指向当前分支；
- 分支名会指向该分支上的最新提交。

连续提交后，提交之间会形成一条历史链：

```mermaid
flowchart TB
    subgraph history[提交历史]
        direction LR
        C1[commit A\n初始版本] --> C2[commit B\n添加训练脚本] --> C3[commit C\n修改配置]
    end

    HEAD{{HEAD}}
    main([main 分支])
    HEAD -.-> main
    main -.-> C3

    classDef commit fill:#eef2ff,stroke:#4f46e5,stroke-width:1px,color:#111827;
    classDef branch fill:#ecfdf5,stroke:#059669,stroke-width:2px,color:#064e3b;
    classDef head fill:#fff7ed,stroke:#ea580c,stroke-width:2px,color:#7c2d12;
    class C1,C2,C3 commit;
    class main branch;
    class HEAD head;
```

上图可以这样理解：`main` 指向最新提交 `commit C`，`HEAD` 指向当前所在的分支 `main`。当再次提交时，会生成新的 `commit D`，然后 `main` 自动移动到 `commit D`。

## 分支 Branch

分支可以理解为指向某个 `commit` 的名字。创建分支时，新分支会先指向当前提交；在这个分支上继续提交后，新分支会移动到新的提交。

例如，从 `main` 创建一个 `FixBug` 分支，并将 `HEAD` 指向该分支：



In [ ]:
git switch -c FixBug




然后在 `FixBug` 上修复问题，使用 `git add` 暂存修改后提交：



In [ ]:
git commit




```mermaid
flowchart TB
    subgraph history[提交历史]
        direction LR
        C1[commit A\n初始版本] --> C2[commit B\n添加功能] --> C3[commit C\n发现问题]
        C3 --> C4[commit D\nFixBug 修复]
    end

    main([main]) -.-> C3
    FixBug([FixBug]) -.-> C4
    HEAD{{HEAD}} -.-> FixBug

    classDef commit fill:#eef2ff,stroke:#4f46e5,stroke-width:1px,color:#111827;
    classDef mainBranch fill:#ecfdf5,stroke:#059669,stroke-width:2px,color:#064e3b;
    classDef fixBranch fill:#eff6ff,stroke:#2563eb,stroke-width:2px,color:#1e3a8a;
    classDef head fill:#fff7ed,stroke:#ea580c,stroke-width:2px,color:#7c2d12;
    class C1,C2,C3,C4 commit;
    class main mainBranch;
    class FixBug fixBranch;
    class HEAD head;
```

图中，`main` 仍然停在 `commit C`，`FixBug` 指向新的 `commit D`。当前分支就是 `HEAD` 指向的分支，因此这里表示当前正在 `FixBug` 分支上工作。

## 合并分支

假设在修改 `FixBug` 分支期间，`main` 分支也发现了一个紧急问题，并提交了一个新的 `commit`。等 `FixBug` 完成后，可以切回 `main`，用 `merge` 合并 `FixBug` 的修改：



In [ ]:
git switch main
git merge --no-edit FixBug




流程如下：

```mermaid
flowchart TB
    subgraph history[提交历史]
        direction LR
        C1[commit A\n初始版本] --> C2[commit B\n添加功能] --> C3[commit C\n发现问题]
        C3 --> M1[commit E\nmain 紧急修复]
        C3 --> F1[commit D\nFixBug 修复]
        M1 --> Merge[commit F\nmerge FixBug]
        F1 --> Merge
    end

    main([main]) -.-> Merge
    FixBug([FixBug]) -.-> F1
    HEAD{{HEAD}} -.-> main

    classDef commit fill:#eef2ff,stroke:#4f46e5,stroke-width:1px,color:#111827;
    classDef mergeCommit fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef mainBranch fill:#ecfdf5,stroke:#059669,stroke-width:2px,color:#064e3b;
    classDef fixBranch fill:#eff6ff,stroke:#2563eb,stroke-width:2px,color:#1e3a8a;
    classDef head fill:#fff7ed,stroke:#ea580c,stroke-width:2px,color:#7c2d12;
    class C1,C2,C3,M1,F1 commit;
    class Merge mergeCommit;
    class main mainBranch;
    class FixBug fixBranch;
    class HEAD head;
```

`merge` 会把两个分支的修改合到一起。上图中，`commit F` 是一次合并提交，它同时连接了 `main` 的紧急修复和 `FixBug` 的修复结果。

## 示例

下面用一个最小项目观察提交、分支和合并：



In [ ]:
mkdir git-principle-demo
cd git-principle-demo
git init -b main

echo 'print("hello")' > app.py
git add app.py
git commit -m "Add app"

git switch -c FixBug
echo 'print("fix bug")' >> app.py
git add app.py
git commit -m "Fix bug"

git switch main
echo '# demo' > README.md
git add README.md
git commit -m "Add readme"

git merge --no-edit FixBug




这个例子中，`main` 和 `FixBug` 从同一个提交出发，各自新增提交，最后通过 `merge` 合到一起。`--no-edit` 表示使用 Git 默认生成的合并说明。

## 查看历史

运行下面的命令查看提交关系：



In [ ]:
git log --oneline --graph --decorate --all




常用选项含义：

- `--oneline`：每个提交显示为一行；
- `--graph`：用字符画出分支和合并关系；
- `--decorate`：显示 `HEAD`、`main`、`FixBug` 等引用；
- `--all`：显示所有分支的历史。

## 练习

打开 [Learn Git Branching](https://learngitbranching.js.org/?locale=zh_CN)，完成“基础篇”的第 1-3 关。

第 4 关作为选做，用来了解 `rebase`。它是一种整理提交历史的高级操作，本章只要求知道它的存在。